# 流式响应

## 学习目标

* 理解流式响应的工作原理
* 掌握如何处理流事件


首先导入 `anthropic` SDK 并设置客户端：

In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

#load environment variable
load_dotenv()

#automatically looks for an "ANTHROPIC_API_KEY" environment variable
client = Anthropic()

到目前为止，我们使用以下语法向 Claude 发送消息：


In [3]:
response = client.messages.create(
    messages=[
        {
            "role": "user",
            "content": "Write me an essay about macaws and clay licks in the Amazon",
        }
    ],
    model="claude-3-haiku-20240307",
    max_tokens=800,
    temperature=0,
)
print("We have a response back!")
print("========================")
print(response.content[0].text)

We have a response back!
Here is an essay about macaws and clay licks in the Amazon:

Macaws and Clay Licks in the Amazon

Deep within the lush, verdant rainforests of the Amazon basin, a remarkable natural phenomenon takes place. Flashes of vibrant color dart through the canopy, as large, magnificent parrots known as macaws congregate at special sites called clay licks. These clay licks, or "collpas" as they are known locally, are essential to the survival and well-being of macaws and other Amazonian wildlife.

Macaws are some of the most striking and iconic birds of the Amazon. With their vividly-hued plumage, hooked beaks, and long, tapered tails, these large parrots are a sight to behold as they soar through the treetops. The most well-known species include the scarlet macaw, the blue-and-gold macaw, and the green-winged macaw, each adorned in a stunning array of reds, blues, greens, and golds. 

These magnificent birds play a crucial role in the Amazon ecosystem, acting as importa

这种写法可以正常工作，但需要记住的是，采用这种方式我们只能在所有内容生成完成后才能获得 API 返回的内容。重新运行上面的单元格，你会看到在整个响应一次性打印出来之前不会有任何输出。在很多情况下这没问题，但如果你的应用需要用户等待整个响应生成完毕才能看到任何内容，这会导致糟糕的用户体验。

**流式响应闪亮登场！**

流式响应使我们能够编写应用程序，在模型生成内容的同时接收内容，而无需等待整个响应生成完毕。这就是 claude.ai 等应用的实现方式。当模型生成响应时，内容会流式传输到用户的浏览器并实时显示：

![claude_streaming.gif](attachment:claude_streaming.gif)

## 处理流式响应

要从 API 获取流式响应，只需要向 `client.messages.create` 传入 `stream=True` 参数即可。这部分很简单。真正需要掌握的是如何处理流式响应以及如何处理传入的数据。

In [4]:
stream = client.messages.create(
    messages=[
        {
            "role": "user",
            "content": "Write me a 3 word sentence, without a preamble.  Just give me 3 words",
        }
    ],
    model="claude-3-haiku-20240307",
    max_tokens=100,
    temperature=0,
    stream=True,
)

让我们看看 `stream` 变量：

In [5]:
stream

看起来没什么内容！这个 stream 对象本身并不能做太多事情。stream 对象是一个生成器对象，会在从 API 收到数据时逐个 yield 服务器发送的事件（Server-Sent Events，简称 SSE）。我们需要编写代码来迭代处理它，并处理每个服务器发送的事件。请记住，我们的数据不再是作为一个最终的数据块传入的。让我们尝试迭代遍历这个流式响应：

In [6]:
for event in stream:
    print(event)

MessageStartEvent(message=Message(id='msg_01EZHjA6qmf6y8VWMZSeBmN5', content=[], model='claude-3-haiku-20240307', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(input_tokens=30, output_tokens=2)), type='message_start')
ContentBlockStartEvent(content_block=ContentBlock(text='', type='text'), index=0, type='content_block_start')
ContentBlockDeltaEvent(delta=TextDelta(text='Cats', type='text_delta'), index=0, type='content_block_delta')
ContentBlockDeltaEvent(delta=TextDelta(text=' me', type='text_delta'), index=0, type='content_block_delta')
ContentBlockDeltaEvent(delta=TextDelta(text='ow lou', type='text_delta'), index=0, type='content_block_delta')
ContentBlockDeltaEvent(delta=TextDelta(text='dly.', type='text_delta'), index=0, type='content_block_delta')
ContentBlockStopEvent(index=0, type='content_block_stop')
MessageDeltaEvent(delta=Delta(stop_reason='end_turn', stop_sequence=None), type='message_delta', usage=MessageDeltaUsage(output_tokens=10))

如你所见，我们从 API 收到了许多服务器发送的事件。让我们仔细看看这些事件代表什么意思。以下是这些事件的彩色编码解释：


![streaming_output.png](attachment:streaming_output.png)

每个流包含一系列按以下顺序排列的事件：
* **MessageStartEvent** - 一条内容为空的消息
* **内容块系列** - 每个内容块包含：
    * 一个 **ContentBlockStartEvent**
    * 一个或多个 **ContentBlockDeltaEvent**
    * 一个 **ContentBlockStopEvent**
* 一个或多个 **MessageDeltaEvent**，用于表示最终消息的顶层变更
* 一个最终的 **MessageStopEvent**

在上面的响应中，只有一个内容块。这张图显示了与之关联的所有事件：

![content_block_streaming.png](attachment:content_block_streaming.png)

我们关心的所有实际模型生成内容都来自 ContentBlockDeltaEvent，它的 type 字段设置为 "content_block_delta"。要获取实际内容，我们需要访问 `delta` 中的 `text` 属性。让我们尝试仅打印生成的内容：

In [7]:
stream = client.messages.create(
    messages=[
        {
            "role": "user",
            "content": "Write me a 3 word sentence, without a preamble.  Just give me 3 words",
        }
    ],
    model="claude-3-haiku-20240307",
    max_tokens=100,
    temperature=0,
    stream=True,
)
for event in stream:
    if event.type == "content_block_delta":
        print(event.delta.text)

Cats
 me
ow lou
dly.


我们成功打印出了内容，尽管格式有点难以阅读。当使用 Python 的 `print()` 函数打印流式文本时，传入两个额外的参数会很有帮助：
* `end=""`：默认情况下，print() 函数会在打印文本末尾添加一个换行符（\n）。但是通过设置 end=""，我们可以指定打印文本后不跟随换行符。这意味着下一个 print() 语句将在同一行继续打印。
* `flush=True`：flush 参数设置为 True 以强制立即将输出写入控制台或标准输出，而不必等待换行符或缓冲区填满。这确保了文本在从流式响应收到时能实时显示。

让我们尝试进行这些修改：

In [8]:
stream = client.messages.create(
    messages=[
        {
            "role": "user",
            "content": "Write me a 3 word sentence, without a preamble.  Just give me 3 words",
        }
    ],
    model="claude-3-haiku-20240307",
    max_tokens=100,
    temperature=0,
    stream=True,
)
for event in stream:
    if event.type == "content_block_delta":
        print(event.delta.text, flush=True, end="")

Cats meow loudly.

对于这么短的文本，流式响应的效果可能不太明显。让我们尝试让模型生成更长的内容：

In [9]:
stream = client.messages.create(
    messages=[
        {
            "role": "user",
            "content": "How do large language models work?",
        }
    ],
    model="claude-3-haiku-20240307",
    max_tokens=1000,
    temperature=0,
    stream=True,
)
for event in stream:
    if event.type == "content_block_delta":
        print(event.delta.text, flush=True, end="")

Large language models like myself work by using deep learning neural networks that are trained on massive amounts of text data. The key aspects are:

1. Neural network architecture - We use large, multi-layer neural networks with many parameters that can learn complex patterns in language.

2. Training data - We are trained on huge corpora of text from the internet, books, articles, and other sources. This allows us to learn the statistical patterns and structures of language.

3. Self-supervised learning - During training, the model learns to predict the next word in a sequence of text, without any explicit labels. This allows it to learn general language understanding.

4. Transfer learning - The knowledge gained during this pre-training can then be fine-tuned for specific tasks like question answering, summarization, translation, etc.

5. Attention mechanisms - Advanced models like transformers use attention to dynamically focus on the most relevant parts of the input when generatin

如果你还没有运行过上面的单元格，现在可以运行一下。你应该会看到文本内容在传入时逐步打印出来！

如我们所见，ContentBlockDeltaEvent 包含模型生成的文本内容。还有许多其他类型的事件，我们需要关注吗？是的！这里有一个简单的例子：

如果我们想要访问有关 token 使用量的信息，需要在两个地方查找：

* `MessageStartEvent` 包含输入（提示词）token 使用量信息
* `MessageDeltaEvent` 包含生成了多少输出 token 的信息

![streaming_tokens.png](attachment:streaming_tokens.png)

让我们更新上面的代码，打印出提示词使用了多少 token 以及模型生成了多少 token：

In [42]:
stream = client.messages.create(
    messages=[
        {
            "role": "user",
            "content": "How do large language models work?",
        }
    ],
    model="claude-3-haiku-20240307",
    max_tokens=1000,
    temperature=0,
    stream=True,
)
for event in stream:
    if event.type == "message_start":
        input_tokens = event.message.usage.input_tokens
        print("MESSAGE START EVENT", flush=True)
        print(f"Input tokens used: {input_tokens}", flush=True)
        print("========================")
    elif event.type == "content_block_delta":
        print(event.delta.text, flush=True, end="")
    elif event.type == "message_delta":
        output_tokens = event.usage.output_tokens
        print("\n========================", flush=True)
        print("MESSAGE DELTA EVENT", flush=True)
        print(f"Output tokens used: {output_tokens}", flush=True)
        

MESSAGE START EVENT
Input tokens used: 14
Large language models like myself work by using deep learning neural networks that are trained on massive amounts of text data. The key aspects are:

1. Neural network architecture - We use large, multi-layer neural networks with many parameters that can learn complex patterns in language.

2. Training data - We are trained on huge corpora of text from the internet, books, articles, and other sources. This allows us to learn the statistical patterns and structures of language.

3. Self-supervised learning - During training, the model learns to predict the next word in a sequence of text, without any explicit labels. This allows it to learn general language understanding.

4. Transfer learning - The knowledge gained during this pre-training can then be fine-tuned for specific tasks like question answering, summarization, translation, etc.

5. Attention mechanisms - Advanced models like transformers use attention to dynamically focus on the most 

### 其他流式事件类型

在使用流时，你可能会遇到一些其他事件类型，包括：

* **Ping 事件** - 流可能包含任意数量的 ping 事件。
* **错误事件** - 你可能会在事件流中偶尔看到错误事件。例如，在高使用量期间，你可能会收到一个 overloaded_error，这在非流式场景下通常对应 HTTP 529。

以下是一个错误事件的示例：

```
event: error
data: {"type": "error", "error": {"type": "overloaded_error", "message": "Overloaded"}}
```

## 首个 token 的时间（TTFT）

使用流式响应的主要原因是改善首个 token 的时间：也就是你或你的用户收到模型生成的第一部分内容所需的时间。
让我们尝试展示流式响应对 TTFT 的影响。

我们先使用非流式方法。让模型生成一段很长的文本，但限制在 500 个 token：

In [4]:
import time
def measure_non_streaming_ttft():
    start_time = time.time()

    response = client.messages.create(
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": "Write mme a long essay explaining the history of the American Revolution",
            }
        ],
        temperature=0,
        model="claude-3-haiku-20240307",
    )

    response_time = time.time() - start_time

    print(f"Time to receive first token: {response_time:.3f} seconds")
    print(f"Time to recieve complete response: {response_time:.3f} seconds")
    print(f"Total tokens generated: {response.usage.output_tokens}")
    
    print(response.content[0].text)

In [50]:
measure_non_streaming_ttft()

Time to receive first token: 4.194 seconds
Time to recieve complete response: 4.194 seconds
Total tokens generated: 500
Here is a long essay explaining the history of the American Revolution:

The American Revolution was a pivotal event in the history of the United States, marking the country's transition from a collection of British colonies to an independent nation. The roots of the revolution can be traced back to the French and Indian War, which was fought between Britain and France from 1754 to 1763. This conflict, which was part of a larger global war, resulted in the British gaining control of much of North America, including the French colonies. However, the war also left Britain with a significant debt, which it sought to recoup by imposing a series of taxes and regulations on its American colonies.

One of the first major events that led to the American Revolution was the Stamp Act, which was passed by the British Parliament in 1765. This act required all printed materials in

现在让我们使用流式方法尝试同样的操作：

In [57]:
def measure_streaming_ttft():
    start_time = time.time()

    stream = client.messages.create(
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": "Write mme a long essay explaining the history of the American Revolution",
            }
        ],
        temperature=0,
        model="claude-3-haiku-20240307",
        stream=True
    )
    have_received_first_token = False
    for event in stream:
        if event.type == "content_block_delta":
            if not have_received_first_token:
                ttft = time.time() - start_time
                have_received_first_token = True
            print(event.delta.text, flush=True, end="")
        elif event.type == "message_delta":
            output_tokens = event.usage.output_tokens
            total_time = time.time() - start_time

    print(f"\nTime to receive first token: {ttft:.3f} seconds", flush=True)
    print(f"Time to recieve complete response: {total_time:.3f} seconds", flush=True)
    print(f"Total tokens generated: {output_tokens}", flush=True)
    


In [58]:
measure_streaming_ttft()

Here is a long essay explaining the history of the American Revolution:

The American Revolution was a pivotal event in the history of the United States, marking the country's transition from a collection of British colonies to an independent nation. The roots of the revolution can be traced back to the French and Indian War, which was fought between Britain and France from 1754 to 1763. This conflict, which was part of a larger global war, resulted in the British gaining control of much of North America, including the French colonies. However, the war also left Britain with a significant debt, which it sought to recoup by imposing a series of taxes and regulations on its American colonies.

One of the first major events that led to the American Revolution was the Stamp Act, which was passed by the British Parliament in 1765. This act required all printed materials in the colonies, including newspapers, pamphlets, bills, legal documents, licenses, almanacs, dice, and playing cards, to 

让我们比较一下结果。

* **不使用流式响应**
    * **收到首个 token 的时间：** 4.194 秒
    * **收到完整响应的时间：** 4.194 秒
    * **生成的 token 总数：** 500
* **使用流式响应**
    * **收到首个 token 的时间：** 0.492 秒
    * **收到完整响应的时间：** 4.274 秒
    * **生成的 token 总数：** 500

如你所见，TTFT 的差异非常显著！这个演示只生成了 500 个 token，而且使用的是 Haiku（我们最快的模型）。如果我们尝试用 Opus 生成 1000 个 token 的例子，数据会截然不同！
    

In [2]:
def compare_ttft():
    def measure_streaming_ttft():
        start_time = time.time()

        stream = client.messages.create(
            max_tokens=1000,
            messages=[
                {
                    "role": "user",
                    "content": "Write mme a very very long essay explaining the history of the American Revolution",
                }
            ],
            temperature=0,
            model="claude-3-opus-20240229",
            stream=True
        )
        have_received_first_token = False
        for event in stream:
            if event.type == "content_block_delta":
                if not have_received_first_token:
                    ttft = time.time() - start_time
                    have_received_first_token = True
            elif event.type == "message_delta":
                output_tokens = event.usage.output_tokens
                total_time = time.time() - start_time
        return (ttft, output_tokens)
    
    def measure_non_streaming_ttft():
        start_time = time.time()

        response = client.messages.create(
            max_tokens=1000,
            messages=[
                {
                    "role": "user",
                    "content": "Write mme a very very long essay explaining the history of the American Revolution",
                }
            ],
            temperature=0,
            model="claude-3-opus-20240229"
        )
        ttft = time.time() - start_time
        return (ttft, response.usage.output_tokens)
    
    streaming_ttft, streaming_tokens = measure_streaming_ttft()
    non_streaming_ttft, non_streaming_tokens = measure_non_streaming_ttft()

    print("OPUS STREAMING")
    print(f"Time to first token: {streaming_ttft}")
    print(f"Tokens generated: {streaming_tokens}")
    print("#########################################################")
    print("OPUS NON STREAMING")
    print(f"Time to first token: {non_streaming_ttft}")
    print(f"Tokens generated: {non_streaming_tokens}")

        

In [5]:
# DO NOT RUN THIS! It takes over a minute to run and generates around 2000 tokens with Opus! 
compare_ttft()

OPUS STREAMING
Time to first token: 1.8863098621368408
Tokens generated: 997
#########################################################
OPUS NON STREAMING
Time to first token: 47.03177309036255
Tokens generated: 998


当我们使用 Opus 生成更长的文本时，流式响应对 TTFT 的影响更加明显。使用非流式方法，花了 47 秒才获得第一个 token。而使用流式响应，只需要 1.8 秒就能获得第一个 token！

**注意：请记住，流式响应并不会神奇地缩短模型生成响应所需的总时间。我们可以更快地获得初始数据，但从请求开始到收到最终生成的 token 仍然需要相同的时间**



## 流式响应辅助工具

Python SDK 为流式消息提供了多种便捷方法。我们可以使用 `client.messages.stream` 而不是使用 `stream=True` 调用 `client.messages.create`。`client.messages.stream()` 返回一个 MessageStreamManager，它是一个上下文管理器，会生成一个可迭代的 MessageStream，可以发出事件并累积消息。

下面的代码使用 `client.messages.stream`，它允许我们使用辅助方法（如 `stream.text_stream`）轻松访问流入的生成文本内容，而无需手动检查流事件类型。`stream.text_stream` 提供了一个迭代器，只遍历流中的文本增量。

还有其他有用的辅助方法，如 `get_final_message`，它会在流被完全读取后返回最终累积的消息。如果你既想使用流式响应，又需要在完成时访问整个完成的文本生成，这个方法会很有用。当然，你可以编写一些代码来构建自己的累积消息，但这个辅助方法让这件事变得简单。

以下示例在收到每个传入的文本片段时打印出来，并在流完成时打印最终完成的消息：


In [79]:
from anthropic import AsyncAnthropic

client = AsyncAnthropic()

async def streaming_with_helpers():
    async with client.messages.stream(
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": "Write me sonnet about orchids",
            }
        ],
        model="claude-3-opus-20240229",
    ) as stream:
        async for text in stream.text_stream:
            print(text, end="", flush=True)

    final_message = await stream.get_final_message()
    print("\n\nSTREAMING IS DONE.  HERE IS THE FINAL ACCUMULATED MESSAGE: ")
    print(final_message.to_json())

await streaming_with_helpers()

In gardens fair, where beauty reigns supreme,
The orchid stands, a queen among the blooms,
Her delicate petals, like a lovely dream,
Adorned in nature's most exquisite plumes.

With colors ranging from pure white to bold,
And patterns intricate, a work of art,
Each blossom tells a story, bright and old,
Of evolution's path, a world apart.

From rainforests dense to mountain peaks so high,
The orchid thrives, a testament to grace,
Her beauty captivates the wandering eye,
And in our hearts, she finds a cherished place.

Oh, orchid fair, your splendor knows no bounds,
Forever in our gardens and hearts you'll be found.

STREAMING IS DONE.  HERE IS THE FINAL ACCUMULATED MESSAGE: 
{
  "id": "msg_018x1nZcs3sfq15zKaS4z4gD",
  "content": [
    {
      "text": "In gardens fair, where beauty reigns supreme,\nThe orchid stands, a queen among the blooms,\nHer delicate petals, like a lovely dream,\nAdorned in nature's most exquisite plumes.\n\nWith colors ranging from pure white to bold,\nAnd patter


当使用 `client.messages.stream()` 时，我们还可以定义自定义事件处理程序，在任何流事件发生时或仅在生成文本时等情况下运行。

下面的示例使用了两个自定义事件处理程序。我们使用 `client.messages.stream()` 并让模型"生成一首 5 个词的诗"。我们定义了自己的 `MyStream` 类，其中包含两个事件处理程序：

* `on_text` - 当文本 ContentBlock 对象正在被累积时触发此事件。第一个参数是文本增量，第二个参数是当前累积的文本。在下面的示例中，我们使用此事件处理程序在文本流入时打印出来。文本以绿色打印以便更容易可视化。
* `on_stream_event` - 当从 API 收到任何事件时触发此事件。在下面的示例中，我们在收到任何事件时打印事件类型。

然后我们向 `client.messages.stream` 传递一个 `event_handler` 参数来注册回调方法，这些方法在某些事件发生时触发：


In [94]:
from anthropic import AsyncAnthropic, AsyncMessageStream

client = AsyncAnthropic()

green = '\033[32m'
reset = '\033[0m'

class MyStream(AsyncMessageStream):
    async def on_text(self, text, snapshot):
        # This runs only on text delta stream messages
        print(green + text + reset, flush=True) #model generated content is printed in green

    async def on_stream_event(self, event):
        # This runs on any stream event
        print("on_event fired:", event.type)

async def streaming_events_demo():
    async with client.messages.stream(
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": "Generate a 5-word poem",
            }
        ],
        model="claude-3-opus-20240229",
        event_handler=MyStream,
    ) as stream:
        # Get the final accumulated message, after the stream is exhausted
        message = await stream.get_final_message()
        print("accumulated final message: ", message.to_json())

await streaming_events_demo()

on_event fired: message_start
on_event fired: content_block_start
on_event fired: content_block_delta
Whis
on_event fired: content_block_delta
pers
on_event fired: content_block_delta
 dance
on_event fired: content_block_delta
,
on_event fired: content_block_delta
 secrets
on_event fired: content_block_delta
 unf
on_event fired: content_block_delta
ol
on_event fired: content_block_delta
d,
on_event fired: content_block_delta
 love
on_event fired: content_block_delta
.
on_event fired: content_block_stop
on_event fired: message_delta
on_event fired: message_stop
accumulated final message:  {
  "id": "msg_014G44rr3M14DzadHXPn9Xaj",
  "content": [
    {
      "text": "Whispers dance, secrets unfold, love.",
      "type": "text"
    }
  ],
  "model": "claude-3-opus-20240229",
  "role": "assistant",
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "input_tokens": 14,
    "output_tokens": 14
  }
}


Python SDK 为我们提供了其他一些可利用的事件处理程序，包括：

##### `on_message(message: Message)`
当完整的消息对象被累积时触发。此事件对应于 message_stop SSE。

##### `on_content_block(content_block: ContentBlock)`
当完整的内容块对象被累积时触发。此事件对应于 content_block_stop SSE。

##### `on_exception(exception: Exception)`
当在流式传输响应时遇到异常时触发。

##### `on_timeout()`
当请求超时时触发。

##### `on_end()`
流中最后一个触发的事件。

***

## 练习

编写一个使用流式响应的简单 Claude 聊天机器人。下面的 gif 展示了这个聊天机器人应该如何工作。请注意，输出的颜色编码完全是可选的，主要是为了使 gif 更容易观看：

![streaming_chat_exercise.gif](attachment:streaming_chat_exercise.gif)

### 参考答案
以下是上述练习的一种简单实现。为了获得最佳体验，请将其作为独立的 Python 脚本运行，而不是在此 notebook 的单元格中运行：

In [ ]:
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# ANSI color codes
BLUE = "\033[94m"
GREEN = "\033[92m"
RESET = "\033[0m"

def chat_with_claude():
    print("Welcome to the Claude Chatbot!")
    print("Type 'quit' to exit the chat.")
    
    conversation = []
    
    while True:
        user_input = input(f"{BLUE}You: {RESET}")
        
        if user_input.lower() == 'quit':
            print("Goodbye!")
            break
        
        conversation.append({"role": "user", "content": user_input})
        
        print(f"{GREEN}Claude: {RESET}", end="", flush=True)
        
        stream = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=1000,
            messages=conversation,
            stream=True
        )
        
        assistant_response = ""
        for chunk in stream:
            if chunk.type == "content_block_delta":
                content = chunk.delta.text
                print(f"{GREEN}{content}{RESET}", end="", flush=True)
                assistant_response += content
        
        print()  # New line after the complete response
        
        conversation.append({"role": "assistant", "content": assistant_response})

if __name__ == "__main__":
    chat_with_claude()

***